In [1]:
import matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Iterable, Tuple

from seaborn import color_palette

import plot_benchmark
from pathlib import Path
import os

%load_ext autoreload
%autoreload
%matplotlib inline

results_rtx2080 = Path("../results/2025-10-25_18-51_Benchmark_Result_RTX2080_without_shr.csv")
results_rtx3080 = Path("../results/RTX3080_Benchmark_Result.csv")
results_rtx4060 = Path("../results/RTX4060_Benchmark_Result.csv")

In [2]:
df_rxt2080 = pd.read_csv(results_rtx2080)
df_rxt2080["GPU"] = "RTX2080"
df_rtx3080 = pd.read_csv(results_rtx3080)
df_rtx3080["GPU"] = "RTX3080"
df_rtx4060 = pd.read_csv(results_rtx4060)
df_rtx4060["GPU"] = "RTX4060"

FileNotFoundError: [Errno 2] No such file or directory: '../results/2025-10-25_18-51_Benchmark_Result_RTX2080_without_shr.csv'

In [ ]:
# Concat the three dataframes
df = pd.concat([df_rxt2080, df_rtx3080, df_rtx4060])
df

In [ ]:
def filter_and_normalize(df: pd.DataFrame,
                         name: str,
                         problem_size: int,
                         runtime: str = "Kernel Time",
                         exclude: str | list[str] = "Cublas") -> pd.DataFrame:
    """
    Filter by Name and Problem Size, exclude frameworks, and compute:
    - Application Efficiency [%]: normalized inverse of chosen runtime column (lower is better -> higher score)
      computed separately within each GPU group.
    """
    df = df.copy()
    required_cols = {"Framework", "Name", "Problem Size", runtime, "GPU"}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Filter by name and problem size
    out = df.loc[(df["Name"] == name) & (df["Problem Size"] == problem_size)].copy()

    # Exclude frameworks
    if exclude:
        excl = [exclude] if isinstance(exclude, str) else list(exclude)
        out = out.loc[~out["Framework"].isin(excl)]

    # Drop columns not needed for the analysis if they exist
    to_drop = ["Name", "Problem Size"]
    existing_drop = [c for c in to_drop if c in out.columns]
    if existing_drop:
        out.drop(columns=existing_drop, inplace=True)

    # Application Efficiency per GPU: normalize inverse within each GPU group (best/fastest in GPU = 100)
    runtime_series = out[runtime].astype(float)
    min_per_gpu = runtime_series.groupby(out["GPU"]).transform("min")
    out["Application Efficiency [%]"] = (min_per_gpu / runtime_series) * 100.0

    return out


def filter_and_concat(
        df: pd.DataFrame,
        queries: Iterable[Tuple[str, int, str | None]],
        exclude: str | list[str] = "Cublas",
) -> pd.DataFrame:
    """
    Run filter_and_normalize multiple times and concatenate results.
    Each query is a tuple of (name, problem_size, runtime). If runtime is None, defaults to "Kernel Time".
    The resulting dataframe will include the original Name and Problem Size for each query.
    """
    frames: list[pd.DataFrame] = []
    for q in queries:
        if len(q) != 3:
            raise ValueError("Each query must be a tuple: (name, problem_size, runtime)")
        name, problem_size, runtime = q
        rt = runtime if runtime is not None else "Kernel Time"
        out = filter_and_normalize(df, name=name, problem_size=problem_size, runtime=rt, exclude=exclude).copy()
        # Re-add the identifying columns
        out["Name"] = name
        out["Problem Size"] = problem_size
        out["Runtime Column"] = rt
        frames.append(out)

    if not frames:
        return pd.DataFrame(columns=["Name", "Problem Size", "Runtime Column"])

    return pd.concat(frames, ignore_index=True)


In [ ]:
filter_and_concat(
    df=df,
    queries=[
        ("MatrixMultiplication", 8192, "Wall Clock Time"),
    ]
)

In [ ]:
frameworks_version = [
    "Alpaka", "OpenMP", "Vulkan", "AdaptiveCpp", "AdaptiveCpp[SharedMemory]",
    "CPP", "OpenCL", "Boost", "Kokkos", "Cublas", "Cuda", "Cuda[SharedMemory]",
    "OpenACC", "Slang-Cuda", "Slang-Vulkan"
]
tab20 = matplotlib.colormaps["tab20"].colors
color_map = {
    "CPP": tab20[0],

    "Vulkan": tab20[2],
    "Slang-Vulkan": tab20[3],

    "Cuda[Naive]": tab20[8],
    "Cuda": tab20[8],
    "Slang-Cuda": tab20[9],
    "Cuda[SharedMemory]": tab20[12],
    "Cublas": tab20[13],

    "AdaptiveCpp[Naive]": tab20[4],
    "AdaptiveCpp": tab20[4],
    "AdaptiveCpp[SharedMemory]": tab20[5],

    "OpenACC": tab20[14],
    "OpenMP": tab20[18],

    "Alpaka": tab20[10],

    "OpenCL": tab20[16],
    "Boost": tab20[17],

    "Kokkos": tab20[6],
}

In [ ]:
matMul_df = filter_and_concat(
    df=df,
    queries=[
        ("MatrixMultiplication", 512, "Wall Clock Time"),
        ("MatrixMultiplication", 1024, "Wall Clock Time"),
        ("MatrixMultiplication", 2048, "Wall Clock Time"),
        ("MatrixMultiplication", 4096, "Wall Clock Time"),
        ("MatrixMultiplication", 8192, "Wall Clock Time"),
    ]
)
matMul_df

Task: Create a seaborn line plot from matMul_df with:
- x: Problem Size
- y: Application Efficiency [%]
- color: Framework[Version] using color_map
- line style: GPU (RTX2080, RTX3080, RTX4060)

In [ ]:
palette = {k: color_map[k] for k in matMul_df["Framework[Version]"].unique() if k in color_map}

plt.figure(figsize=(8, 5))
ax = sns.lineplot(
    data=matMul_df,
    x="Problem Size",
    y="Application Efficiency [%]",
    hue="Framework[Version]",
    style="GPU",
    markers=True,
    dashes=True,
    palette=palette
)
ax.set_xscale("log")
ax.set_ylim(0, 105)
ax.set_xlabel("Problem Size")
ax.set_ylabel("Application Efficiency [%]")
ax.grid(True, linestyle="--")
plt.tight_layout()
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
matMul_df_8192 = matMul_df.loc[matMul_df["Problem Size"] == 8192]
palette = {k: color_map[k] for k in matMul_df["Framework[Version]"].unique() if k in color_map}

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=matMul_df_8192,
    x="Framework[Version]",
    y="Application Efficiency [%]",
    hue="GPU",
    palette="tab20",
    ci=None
)
ax.set_ylim(0, 105)
ax.set_xlabel("Framework[Version]")
ax.set_ylabel("Application Efficiency [%]")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
nbody_df = filter_and_concat(
    df=df,
    queries=[
        ("NBody", 100, "Wall Clock Time"),
        ("NBody", 1000, "Wall Clock Time"),
        ("NBody", 10000, "Wall Clock Time"),
    ]
)

In [ ]:
palette = {k: color_map[k] for k in nbody_df["Framework[Version]"].unique() if k in color_map}

plt.figure(figsize=(8, 5))
ax = sns.lineplot(
    data=nbody_df,
    x="Problem Size",
    y="Application Efficiency [%]",
    hue="Framework[Version]",
    style="GPU",
    markers=True,
    dashes=True,
    palette=palette
)
ax.set_xscale("log")
ax.set_ylim(0, 105)
ax.set_xlabel("Problem Size")
ax.set_ylabel("Application Efficiency [%]")
ax.grid(True, linestyle="--")
plt.tight_layout()
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
nbody_df_1k = nbody_df.loc[nbody_df["Problem Size"] == 1000]
palette = {k: color_map[k] for k in nbody_df["Framework[Version]"].unique() if k in color_map}

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=nbody_df_1k,
    x="Framework[Version]",
    y="Application Efficiency [%]",
    hue="GPU",
    palette="tab20",
    ci=None
)
ax.set_ylim(0, 105)
ax.set_xlabel("Framework[Version]")
ax.set_ylabel("Application Efficiency [%]")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.gcf().set_dpi(300)
plt.show()

In [ ]:
nbody_df_10k = nbody_df.loc[nbody_df["Problem Size"] == 10000]
palette = {k: color_map[k] for k in nbody_df["Framework[Version]"].unique() if k in color_map}

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=nbody_df_10k,
    x="Framework[Version]",
    y="Application Efficiency [%]",
    hue="GPU",
    palette="tab20",
    ci=None
)
ax.set_ylim(0, 105)
ax.set_xlabel("Framework[Version]")
ax.set_ylabel("Application Efficiency [%]")
ax.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.gcf().set_dpi(300)
plt.show()